In [ ]:
CONFIG = {
    "seq_len": 10,
    "batch_size": 4,
    "embed_size": 6,
    "num_heads": 2, # TODO: assert num_heads / embed_size
    #"num_layers": 2,
    #"dropout": 0.1,
}


Load dataset:

In [ ]:
import requests
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATASET_TEXT = requests.get(url).text
DATASET_TEXT

Find tokens:

In [ ]:
chars = sorted(list(set(DATASET_TEXT)))
chars

Create tokenizer:

In [ ]:
ctoi = {c:i for i, c in enumerate(chars)}
itoc = {i:c for i, c in enumerate(chars)}
encode = lambda text: [ctoi[c] for c in text]
decode = lambda tokens: "".join([itoc[i] for i in tokens])
decode(encode("hello world"))

Create token embedding table:

In [ ]:
import torch
import torch.nn as nn

vocab_size = len(ctoi)
embed_size = CONFIG["embed_size"]
CONFIG["vocab_size"] = vocab_size

seq_len = CONFIG["seq_len"]
pad = lambda text: " " * max(0, seq_len - len(text)) + text
text = pad("hello")
tokens = torch.tensor(encode(text))
token_embedding_table = nn.Embedding(vocab_size, embed_size)
tokens_emb = token_embedding_table(tokens)
tokens_emb.shape, tokens_emb

Create position embedding table:

In [ ]:
seq_len = CONFIG["seq_len"]
embed_size = CONFIG["embed_size"]

position_embedding_table = nn.Embedding(seq_len, embed_size)
positions = torch.arange(seq_len)
positions_emb = position_embedding_table(positions)
positions_emb.shape, positions_emb

Test combining token embeddings and positions embeddings:

In [ ]:
x = tokens_emb + positions_emb
x.shape, x

Create batch sampler:

In [ ]:
DATASET_TOKENS = encode(DATASET_TEXT)

def sample_batch(batch_len):
    seq_len = CONFIG["seq_len"]
    sequences = []
    for _ in range(batch_len):
        offset_idx = torch.randint(len(DATASET_TOKENS) - seq_len, (1,))
        sequence = DATASET_TOKENS[offset_idx:offset_idx+seq_len]
        sequences.append(sequence)
    return torch.tensor(sequences)

batch = sample_batch(10)
batch.shape, batch


Test decoding batch:

In [ ]:
for i in range(batch.shape[0]):
    tokens = batch[i].tolist()
    text = decode(tokens)
    print(f"sequence {i}: {text}")

In [ ]:
batch_size = CONFIG["batch_size"]
seq_len = CONFIG["seq_len"]
x = sample_batch(batch_size)
x_emb = token_embedding_table(x)
positions = torch.arange(seq_len)
pos_emb = position_embedding_table(positions)
x = x_emb + pos_emb
x.shape, x[0][0]

In [ ]:
qkv_shape = (CONFIG["embed_size"], CONFIG["embed_size"]) # TODO; call embed_dim
Wq = torch.randn(qkv_shape)
Wk = torch.randn(qkv_shape)
Wv = torch.randn(qkv_shape)

In [ ]:
Q = x @ Wq
K = x @ Wk
V = x @ Wv
Q.shape, K.shape, V.shape

In [ ]:
Kt = K.transpose(-2, -1)
Kt.shape

In [ ]:
QKt = Q @ Kt # (B, T, C) @ (B, C, T) -> (B, T, T)
QKt.shape, QKt[0][0]

In [ ]:
QKt_scaled = QKt / torch.sqrt(torch.tensor(embed_size))
QKt_scaled.shape, QKt_scaled[0][0]

In [ ]:
tril = torch.tril(torch.ones((seq_len, seq_len)))
tril

In [ ]:
mask = tril == 0
mask

In [ ]:
QKt_scaled_masked = QKt_scaled.masked_fill(mask, float("-inf"))
QKt_scaled_masked.shape, QKt_scaled_masked[0]

In [ ]:
from torch.nn import functional as F
import matplotlib.pyplot as plt

attention = F.softmax(QKt_scaled_masked, dim=-1)

plt.imshow(attention[0].detach().cpu(), cmap='viridis')
plt.colorbar()
plt.title("Attention Map")
plt.xlabel("Key positions")
plt.ylabel("Query positions")
plt.show()

In [ ]:
output = attention @ V
output.shape

In [ ]:
ffn1 = nn.Linear(embed_size, embed_size * 4)
gelu = nn.GELU()
ffn2 = nn.Linear(embed_size * 4, embed_size)
n_heads = CONFIG["num_heads"]
head_size = embed_size // n_heads
lm_head = nn.Linear(embed_size, head_size)

In [ ]:
output_head = lm_head(ffn2(gelu(ffn1(output))))
output_head.shape, output_head

In [ ]:
class AttentionHead(nn.Module):
    def __init__(self):
        vocab_size = CONFIG["vocab_size"]
        embed_size = CONFIG["embed_size"]
        
        att_matrix_shape = (embed_size, embed_size)
        self.token_embedding_table = nn.Embedding(vocab_size, embed_size)
        self.position_embedding_table = nn.Embedding(seq_len, embed_size)
        self.Wq = torch.randn(att_matrix_shape)
        self.Wk = torch.randn(att_matrix_shape)
        self.Wv = torch.randn(att_matrix_shape)

    def forward(self, x):
        seq_len = CONFIG["seq_len"]
        embed_size = CONFIG["embed_size"]

        _x_emb = self.token_embedding_table(x)
        _pos_emb = self.position_embedding_table(torch.arange(seq_len))
        x_emb = _x_emb + _pos_emb

        Q = x_emb @ Wq
        K = x_emb @ Wk
        V = x_emb @ Wv
        Kt = K.transpose(-2, -1)
        QKt = Q @ Kt
        QKt_scaled = QKt / torch.sqrt(embed_size)
        tril = torch.tril(torch.ones(seq_len, seq_len))
        mask = tril == 0
        QKt_masked = QKt_scaled.mask_fill(mask, float("-inf"))
        attention = F.softmax(QKt_masked, dim=-1) # TODO: confirm this
        out = attention @ V

        return out


x = sample_batch(1)
AttentionHead()(x)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self):
        n_heads = CONFIG["num_heads"]
        embedding_size = CONFIG["embedding_size"]
        head_size = embedding_size / n_heads
        self.heads = nn.ModuleList([AttentionHead(head_size) for _ in range(n_heads)])

    def forward(self, x):
        out = self.heads(x)
        out = torch.concat(out, dim=-1)

x = sample_batch(1)
MultiHeadAttention()(x)

In [ ]:
class Block(nn.Module):
    def __init__(self):
        self.head = MultiHeadAttention()

    def forward(self, x):
        x = x + self.head(x)
        # TODO: what here?
        return x


In [ ]:
class Transformer(nn.Module):
    def __init__(self):
        # TODO: seq vs parallel
        self.blocks = nn.ModuleList([Block() for _ in range(CONFIG["num_layers"])])
        self.lm_head = nn.Linear(CONFIG["embedding_size"], CONFIG["vocab_size"])

    def forward(self, x):
        x = self.blocks(x)
        x = self.lm_head(x)
        return x